# Step 1: Environment Verification & Basic Data Loading

In this step, we will import our core libraries, load your dataset file, and check its basic dimensions to verify that your Python environment can read the table perfectly.

In [9]:
import pandas as pd
import numpy as np

# 1. Define your dataset path
# Replace 'path_to_your_dataset.csv' with your actual file name or path
file_path = 'EEG.machinelearing_data_BRMH.csv' 

try:
    # 2. Load the data
    print("Attempting to load the dataset...")
    df = pd.read_csv(file_path)
    
    # 3. Print basic diagnostics
    print("\n[SUCCESS] Dataset loaded successfully!")
    print(f"Total Rows (Samples): {df.shape[0]}")
    print(f"Total Columns (Features): {df.shape[1]}")
    
    # Peek at the target distribution to make sure it matches your profile
    if 'main.disorder' in df.columns:
        print("\nTarget Class Distribution:")
        print(df['main.disorder'].value_counts())
    else:
        print("\n[WARNING] 'main.disorder' column not found. Please double-check your column names.")

except FileNotFoundError:
    print(f"\n[ERROR] File not found at '{file_path}'. Please verify the path or file name.")
except Exception as e:
    print(f"\n[ERROR] An unexpected error occurred: {e}")

Attempting to load the dataset...

[SUCCESS] Dataset loaded successfully!
Total Rows (Samples): 945
Total Columns (Features): 1149

Target Class Distribution:
main.disorder
Mood disorder                         266
Addictive disorder                    186
Trauma and stress related disorder    128
Schizophrenia                         117
Anxiety disorder                      107
Healthy control                        95
Obsessive compulsive disorder          46
Name: count, dtype: int64


# Step 2: Column Selection, Handling 'Sex', and Target Encoding

Drop metadata columns that don't belong in the training data (no., eeg.date, and specific.disorder).

Convert the sex column from text ('M', 'F') to numeric (1, 0) before we touch any imputer.

Encode the main.disorder target text into clean integers (0 through 6) using Scikit-Learn's LabelEncoder.

In [11]:
from sklearn.preprocessing import LabelEncoder

# 1. Define explicit metadata columns to drop
metadata_to_drop = ['no.', 'eeg.date', 'specific.disorder']
columns_to_drop = [col for col in metadata_to_drop if col in df.columns]

# Separate features (X) and target label (y)
X_raw = df.drop(columns=columns_to_drop + ['main.disorder'], errors='ignore')
y_raw = df['main.disorder']

# 2. Convert 'sex' column strings into numbers safely
# This fixes our previous bug by making sure the column is completely numeric
if 'sex' in X_raw.columns:
    X_raw['sex'] = X_raw['sex'].map({'M': 1, 'F': 0})
    print("[SUCCESS] 'sex' column successfully mapped to integers (M=1, F=0).")
else:
    print("[WARNING] 'sex' column not found in features.")

# 3. Encode the categorical targets (the 7 main disorders) into integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)

print("[SUCCESS] Target column 'main.disorder' successfully encoded.")
print("\nClass Mapping Registry:")
for index, class_name in enumerate(label_encoder.classes_):
    print(f"Class {index} -> {class_name}")

# Double check that X_raw contains NO object/string columns anymore
object_cols = X_raw.select_dtypes(include=['object']).columns.tolist()
print(f"\nRemaining object/string columns in features: {object_cols}")

[SUCCESS] 'sex' column successfully mapped to integers (M=1, F=0).
[SUCCESS] Target column 'main.disorder' successfully encoded.

Class Mapping Registry:
Class 0 -> Addictive disorder
Class 1 -> Anxiety disorder
Class 2 -> Healthy control
Class 3 -> Mood disorder
Class 4 -> Obsessive compulsive disorder
Class 5 -> Schizophrenia
Class 6 -> Trauma and stress related disorder

Remaining object/string columns in features: []


# Step 3: Imputation & Explicit Biomarker Feature Engineering

Impute the missing values inside the education and IQ columns using the median value of those columns.

Engineer our two custom, domain-specific biomarkers directly from the pre-calculated powers in your dataset so that our upcoming ML models can actively learn from them.

Frontal Alpha Asymmetry (FAA): Right Frontal Alpha (AB.C.alpha.f.F4) minus Left Frontal Alpha (AB.C.alpha.d.F3).

Theta/Beta Ratio (TBR): Midline Central Theta (AB.B.theta.j.Cz) divided by Midline Central Beta (AB.D.beta.j.Cz).

In [13]:
from sklearn.impute import SimpleImputer

# 1. Clean out the completely empty column before imputing
if 'Unnamed: 122' in X_raw.columns:
    X_raw = X_raw.drop(columns=['Unnamed: 122'])
    print("[INFO] Dropped completely empty 'Unnamed: 122' column.")

# 2. Run the Median Imputer safely now that dimensions are perfectly aligned
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X_raw), columns=X_raw.columns)
print("[SUCCESS] Missing values inside 'education' and 'IQ' columns imputed successfully.")

# 3. Extract and Engineer Domain Biomarkers
# Feature A: Frontal Alpha Asymmetry (FAA)
f3_col = 'AB.C.alpha.d.F3'
f4_col = 'AB.C.alpha.f.F4'

if f3_col in X_imputed.columns and f4_col in X_imputed.columns:
    X_imputed['BIOMARKER_frontal_alpha_asymmetry'] = X_imputed[f4_col] - X_imputed[f3_col]
    print("[SUCCESS] Engineered Feature: Frontal Alpha Asymmetry (FAA) calculated successfully.")
else:
    print("[WARNING] Could not locate F3 or F4 Alpha columns for FAA engineering.")

# Feature B: Theta/Beta Ratio (TBR) at Central Midline (Cz)
theta_cz = 'AB.B.theta.j.Cz'
beta_cz = 'AB.D.beta.j.Cz'

if theta_cz in X_imputed.columns and beta_cz in X_imputed.columns:
    X_imputed['BIOMARKER_theta_beta_ratio_cz'] = X_imputed[theta_cz] / X_imputed[beta_cz]
    print("[SUCCESS] Engineered Feature: Theta/Beta Ratio (TBR) at Cz calculated successfully.")
else:
    print("[WARNING] Could not locate Cz Theta or Beta columns for TBR engineering.")

# 4. Print final sanity check on dimensions
print(f"\nFinal Preprocessed Feature Matrix (X_imputed) shape: {X_imputed.shape}")
print(f"Any remaining missing values left in the dataset: {X_imputed.isna().sum().sum()}")

[INFO] Dropped completely empty 'Unnamed: 122' column.
[SUCCESS] Missing values inside 'education' and 'IQ' columns imputed successfully.
[SUCCESS] Engineered Feature: Frontal Alpha Asymmetry (FAA) calculated successfully.
[SUCCESS] Engineered Feature: Theta/Beta Ratio (TBR) at Cz calculated successfully.

Final Preprocessed Feature Matrix (X_imputed) shape: (945, 1146)
Any remaining missing values left in the dataset: 0


1. Frontal-Parietal Coherence (The Cognitive Control Network)
This tracks the communication between the front of the brain (executive function) and the back/upper part (sensory/spatial processing). It is highly altered in Schizophrenia, Mood disorders, and OCD.

Target Bands: Alpha and Beta are the most predictive here.

2. Left-Right Interhemispheric Coherence (Brain Symmetry)
This measures whether the left and right sides of your brain are talking to each other synchronously. Healthy brains show high symmetry. Drops in interhemispheric coherence (especially in the Frontal Pole FP1-FP2 or Temporal lobes T3-T4) are strong indicators of Anxiety disorders and Trauma/PTSD.

3. Frontal-Occipital Coherence (Long-range Connectivity)
This measures connectivity from the very front (FP1/FP2) to the very back visual cortex (O1/O2). Variations here are often linked to Addictive disorders and Trauma, indicating disrupted long-range brain processing networks.

In [15]:
print("Engineering additional psychiatric coherence features...")

# 1. Frontal-Parietal Alpha Coherence (F3 to P3 - Left Hemisphere Network)
f_p_coh = 'COH.C.alpha.d.F3.n.P3' # Alpha coherence between F3 and P3
if f_p_coh in X_imputed.columns:
    X_imputed['BIOMARKER_frontal_parietal_alpha_coh'] = X_imputed[f_p_coh]
    print("[SUCCESS] Added: Frontal-Parietal Alpha Coherence.")

# 2. Interhemispheric Frontal Pole Beta Coherence (FP1 to FP2)
# High values indicate balanced frontal activation; altered in Anxiety/Stress
fp_inter_coh = 'COH.D.beta.a.FP1.b.FP2'
if fp_inter_coh in X_imputed.columns:
    X_imputed['BIOMARKER_interhemispheric_frontal_beta_coh'] = X_imputed[fp_inter_coh]
    print("[SUCCESS] Added: Interhemispheric Frontal Pole Beta Coherence.")

# 3. Long-Range Frontal-Occipital Delta Coherence (FP1 to O1)
# Often tracked in structural network disruptions like Schizophrenia or Trauma
long_range_coh = 'COH.A.delta.a.FP1.r.O1'
if long_range_coh in X_imputed.columns:
    X_imputed['BIOMARKER_long_range_frontal_occipital_delta_coh'] = X_imputed[long_range_coh]
    print("[SUCCESS] Added: Long-Range Frontal-Occipital Delta Coherence.")

print(f"\nUpdated Preprocessed Feature Matrix (X_imputed) shape: {X_imputed.shape}")

Engineering additional psychiatric coherence features...
[SUCCESS] Added: Frontal-Parietal Alpha Coherence.
[SUCCESS] Added: Interhemispheric Frontal Pole Beta Coherence.
[SUCCESS] Added: Long-Range Frontal-Occipital Delta Coherence.

Updated Preprocessed Feature Matrix (X_imputed) shape: (945, 1149)


# Step 4: Feature Scaling & Stratified Train-Test Splitting

Because this dataset is wide and has slight class imbalances (e.g., 266 Mood Disorder samples vs. 46 OCD samples), we must use a Stratified Split. This ensures that every single split contains an exact, mathematically balanced percentage of all 7 disorders, preventing the model from ignoring the smaller classes.

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Standardize the Numerical Features
# This scales every feature to have a mean of 0 and a variance of 1, which helps trees split cleanly
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)
print("[SUCCESS] Feature matrix standard scaling complete.")

# 2. Perform Stratified Train-Test Split (80% Train, 20% Test)
# stratify=y_encoded is the crucial parameter that fixes class imbalances across sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, 
    y_encoded, 
    test_size=0.20, 
    stratify=y_encoded, 
    random_state=42
)

print("\n=== Split Integrity Check ===")
print(f"Training Set Samples (X_train): {X_train.shape[0]} rows, {X_train.shape[1]} columns")
print(f"Testing Set Samples (X_test): {X_test.shape[0]} rows, {X_test.shape[1]} columns")

# Verify that the test set has an even distribution
unique, counts = np.unique(y_test, return_counts=True)
print("\nTest Set Class Balances (Should match your original distribution ratios):")
for cl, cnt in zip(unique, counts):
    print(f"Class {cl}: {cnt} samples")

[SUCCESS] Feature matrix standard scaling complete.

=== Split Integrity Check ===
Training Set Samples (X_train): 756 rows, 1149 columns
Testing Set Samples (X_test): 189 rows, 1149 columns

Test Set Class Balances (Should match your original distribution ratios):
Class 0: 37 samples
Class 1: 22 samples
Class 2: 19 samples
Class 3: 53 samples
Class 4: 9 samples
Class 5: 23 samples
Class 6: 26 samples


# Step 5: Training the XGBoost Multi-Class Classifier

Now we get to train the core model. Because we have many columns and a complex, multi-class distribution, we will configure an XGBoost Classifier with parameters carefully selected to prevent overfitting:

max_depth=4 to keep individual decision paths constrained.

learning_rate=0.05 for smooth gradient descent updates.

colsample_bytree=0.6 to ensure individual trees only look at a random 60% of columns per split, keeping them from over-indexing on noise.

In [18]:
from xgboost import XGBClassifier
import time

# 1. Initialize the XGBoost Model with optimal dimensions
print("Initializing XGBoost Multi-Class Classifier...")
xgb_model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.6,
    objective='multi:softprob',  # Outputs probability profiles per class
    eval_metric='mlogloss',      # Multiclass log-loss evaluation
    random_state=42,
    n_jobs=-1                   # Utilizes all available processing cores
)

# 2. Train the model and measure processing velocity
print("Starting model training on training partition...")
start_time = time.time()
xgb_model.fit(X_train, y_train)
end_time = time.time()

print(f"[SUCCESS] Model training complete in {end_time - start_time:.2f} seconds!")

Initializing XGBoost Multi-Class Classifier...
Starting model training on training partition...
[SUCCESS] Model training complete in 33.93 seconds!
